# FRED CMT Yields — Exploration Notebook

This notebook fetches the six constant-maturity Treasury yield series from FRED
and builds the key plots you will need to understand before writing `fetch_fred_yields`.

Series pulled:
- DGS3MO — 3-month T-bill
- DGS1   — 1-year
- DGS2   — 2-year
- DGS5   — 5-year
- DGS10  — 10-year
- DGS30  — 30-year

In [ ]:
import requests
import io
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

SERIES = {
    "DGS3MO": 0.25,
    "DGS1":   1.0,
    "DGS2":   2.0,
    "DGS5":   5.0,
    "DGS10":  10.0,
    "DGS30":  30.0,
}
START = "2000-01-01"
END   = "2024-12-31"

def fetch_fred_csv(series_id: str) -> pd.Series:
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text), index_col=0, parse_dates=True)
    s = pd.to_numeric(df.iloc[:, 0], errors="coerce")
    s.name = series_id
    return s.loc[START:END]

frames = [fetch_fred_csv(sid) for sid in SERIES]
raw = pd.concat(frames, axis=1)
raw.index.name = "date"
raw.dropna(how="all", inplace=True)

print(raw.shape)
raw.tail()

## Plot 1 — All Six Yields Over Time

This is the most important plot to internalize. Each line is one maturity.
Notice: yields move mostly together (that's the **level factor**), but the
spread between short and long end changes (that's the **slope factor**).

Key events to spot:
- **2007-2009**: Fed slashed short rates to zero → massive steepening
- **2013 taper tantrum**: 10Y jumped ~100bp in weeks
- **2020 COVID**: Fed cut to zero again overnight
- **2022**: fastest Fed hiking cycle in 40 years → all rates surged

In [ ]:
fig, ax = plt.subplots()
for col in raw.columns:
    ax.plot(raw.index, raw[col], linewidth=1.2, label=col)

ax.set_title("US Treasury CMT Yields 2000–2024")
ax.set_ylabel("Yield (%)")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## Plot 2 — The Yield Curve Shape on a Single Day

This is what "the yield curve" means visually: yield plotted against maturity
on one specific date. The x-axis is maturity in years; y-axis is the yield.

A **normal** curve slopes upward — longer maturities pay more because you're
locking up money longer. An **inverted** curve (short rates > long rates)
historically precedes recessions.

We'll snapshot three dates to compare shapes:

In [ ]:
maturities = list(SERIES.values())
snapshots = {
    "2021-01-04 (steep, post-COVID)":    "2021-01-04",
    "2023-07-03 (inverted, post-hikes)": "2023-07-03",
    "2007-06-01 (normal, pre-GFC)":      "2007-06-01",
}

fig, ax = plt.subplots()
for label, date in snapshots.items():
    row = raw.loc[date].values
    ax.plot(maturities, row, marker="o", linewidth=2, label=label)

ax.set_title("Yield Curve Shape — Three Dates Compared")
ax.set_xlabel("Maturity (years)")
ax.set_ylabel("Yield (%)")
ax.set_xticks(maturities)
ax.legend()
plt.tight_layout()
plt.show()

## Plot 3 — The 10Y–2Y Spread (Slope of the Curve)

The 10Y minus 2Y spread is the most-watched measure of curve steepness.
- Positive → normal curve (longs yield more than shorts)
- Negative → **inverted** curve

Inversions reliably precede recessions (shaded regions below).
This is a proxy for the **slope / PC2 factor** you'll compute in Week 4.

In [ ]:
spread = raw["DGS10"] - raw["DGS2"]
spread = spread.dropna()

fig, ax = plt.subplots()
ax.plot(spread.index, spread, color="steelblue", linewidth=1.2, label="10Y – 2Y spread")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.fill_between(spread.index, spread, 0, where=(spread < 0),
                color="tomato", alpha=0.3, label="Inverted")
ax.fill_between(spread.index, spread, 0, where=(spread >= 0),
                color="steelblue", alpha=0.15)
ax.set_title("10Y – 2Y Yield Spread (Curve Slope) 2000–2024")
ax.set_ylabel("Spread (percentage points)")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

print(f"Days inverted: {(spread < 0).sum()}  ({(spread < 0).mean():.1%} of days)")

## Plot 4 — Daily Yield Changes (What PCA Will Decompose)

PCA in Week 4 doesn't operate on yield *levels* — it operates on day-over-day
yield *changes* in basis points (1 bp = 0.01%).

This plot shows how volatile each maturity is. Notice:
- Short end is choppier in absolute terms during Fed hiking cycles
- Long end moves more smoothly
- But all maturities spike together during crises — that's PC1 (level) dominating

In [ ]:
changes_bps = raw.diff() * 100   # convert pct → basis points

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for i, col in enumerate(raw.columns):
    axes[i].plot(changes_bps.index, changes_bps[col], linewidth=0.6, color="steelblue")
    axes[i].axhline(0, color="black", linewidth=0.5)
    axes[i].set_title(f"{col} daily change (bp)")
    axes[i].set_ylabel("bp")
    axes[i].xaxis.set_major_locator(mdates.YearLocator(4))
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle("Daily Yield Changes by Maturity (basis points)", fontsize=13)
plt.tight_layout()
plt.show()

print("Annualised volatility (bp):")
print((changes_bps.std() * (252**0.5)).round(1))

## Plot 5 — Correlation Between Maturities

If all yields moved independently we'd need a separate model for each.
The correlation matrix shows why PCA works: everything is highly correlated.
Adjacent maturities are nearly perfectly correlated (>0.95).
Even 3M vs 30Y — opposite ends of the curve — are positively correlated (~0.6–0.7).

This high correlation is precisely what PCA collapses into 3 factors.

In [ ]:
import numpy as np

corr = changes_bps.dropna().corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, vmin=0, vmax=1, cmap="RdYlGn")
plt.colorbar(im, ax=ax)

labels = list(corr.columns)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45)
ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}",
                ha="center", va="center", fontsize=8)

ax.set_title("Correlation of Daily Yield Changes")
plt.tight_layout()
plt.show()

## Plot 6 — Yield Curve as a Heatmap Over Time

This combines Plots 1 and 2 into one image. Time runs left-to-right,
maturity runs bottom-to-top, and colour shows the yield level.

You can visually see:
- The 2008 low-rate era (everything blue)
- The 2022 hike cycle (everything red)
- The persistent **inversion** at the short end in 2022-2023 (short end redder than long end)

In [ ]:
filled = raw.ffill().dropna()

# Resample to monthly so the heatmap isn't too dense
monthly = filled.resample("ME").last()

# Build matrix: rows = maturities, cols = dates
mat_labels = list(SERIES.keys())
Z = monthly[mat_labels].T.values  # shape (6, n_months)
dates = monthly.index

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(Z, aspect="auto", cmap="RdYlBu_r",
               extent=[mdates.date2num(dates[0]), mdates.date2num(dates[-1]),
                       -0.5, len(mat_labels) - 0.5])
plt.colorbar(im, ax=ax, label="Yield (%)")
ax.set_yticks(range(len(mat_labels)))
ax.set_yticklabels(mat_labels)
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_title("US Treasury Yield Curve Heatmap 2000–2024 (monthly)")
plt.tight_layout()
plt.show()

## Summary — What to take away

| Plot | What it shows | Why it matters |
|------|---------------|----------------|
| 1. Time series | All 6 yields over 25 years | Yields mostly move together (level factor) |
| 2. Cross-section | Curve shape on a single day | Normal vs inverted; what "the yield curve" means |
| 3. 10Y–2Y spread | Slope of the curve over time | Proxy for PC2; inversions precede recessions |
| 4. Daily changes | Bp moves per maturity | Raw input to PCA; higher vol at short end during hikes |
| 5. Correlation | How tightly maturities co-move | Explains why 3 PCA factors capture 99% of variance |
| 6. Heatmap | Yield level by maturity and time | See inversion regimes visually |

When you write `fetch_fred_yields`, its output feeds directly into these plots
as a sanity check that your pipeline is correct.